<a href="https://colab.research.google.com/github/dinanrzki/Junior-Data-Analyst-Project/blob/main/Marketing_Campaign_Uji_Normalitas_%2B_Mann_Whitney_U.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import re
import warnings
from scipy.stats import shapiro, mannwhitneyu, norm
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# --- A. Script Python - Data Cleaning & Standardization ---

df = pd.read_csv('marketing_campaign_for_vinix.csv')

def standardize_promo(p):
    if pd.isna(p):
        return np.nan
    p2 = str(p).upper().strip().replace("-", " ").replace(" ", "")
    if 'CASHBACK' in p2 or 'CB' in p2:
        return 'CASHBACK 50K'
    if 'DISKON' in p2 or 'DISCOUNT' in p2:
        return 'DISKON 20%'
    return np.nan

def clean_revenue(s):
    if pd.isna(s):
        return np.nan
    c = re.sub(r'[^0-9]', '', str(s))
    return int(c) if c else np.nan

df['Promo_Active'] = df['Promo_Active'].apply(standardize_promo)
df['Daily_Revenue'] = df['Daily_Revenue'].apply(clean_revenue)
df['Tanggal'] = pd.to_datetime(df['Tanggal'])

# IQR Outlier Removal
def iqr_bounds(col):
    q1, q3 = col.quantile(0.25), col.quantile(0.75)
    return q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)

lo_t, hi_t = iqr_bounds(df['Website_Traffic'])
lo_a, hi_a = iqr_bounds(df['Daily_Ad_Spend'].dropna())

df_clean = df[
    (df['Website_Traffic'] >= lo_t) & (df['Website_Traffic'] <= hi_t) &
    (df['Daily_Ad_Spend'] >= lo_a) & (df['Daily_Ad_Spend'] <= hi_a)
].dropna(subset=['Daily_Ad_Spend']).copy()

print(f"Data bersih: {len(df_clean)} baris")


# --- B. Script Python - Task A: Uji Normalitas + Mann-Whitney U ---

diskon = df_clean[df_clean['Promo_Active'] == 'DISKON 20%']['Daily_Revenue']
cashback = df_clean[df_clean['Promo_Active'] == 'CASHBACK 50K']['Daily_Revenue']

# Uji Normalitas Shapiro-Wilk
stat_d, p_d = shapiro(diskon.sample(500, random_state=42))
stat_c, p_c = shapiro(cashback.sample(470, random_state=42))

print(f"Shapiro Diskon 20% : W={stat_d:.4f}, p={p_d:.4f}")
print(f"Shapiro Cashback   : W={stat_c:.4f}, p={p_c:.4f}")

# Uji Mann-Whitney U (Alpha = 0.05)
stat_mw, p_mw = mannwhitneyu(diskon, cashback, alternative='two-sided')

print(f"Mann-Whitney U : {stat_mw:.0f}")
print(f"p-value        : {p_mw:.6e}")
print(f"Keputusan      : {'TOLAK H0' if p_mw < 0.05 else 'GAGAL TOLAK H0'}")


# --- C. Script Python - Task B: Regresi Linier OLS ---

reg_df = df_clean.dropna(subset=['Daily_Ad_Spend', 'Website_Traffic'])
slope, intercept, r, p_reg, se = stats.linregress(
    reg_df['Daily_Ad_Spend'], reg_df['Website_Traffic']
)

r2 = r ** 2
print(f"Intercept   : {intercept:.2f}")
print(f"Slope       : {slope:.8f}")
print(f"R-squared   : {r2:.4f}")
print(f"p-value     : {p_reg:.6e}")
print(f"Per Rp 1 Jt : {slope * 1000000:.0f} pengunjung")

Data bersih: 1023 baris
Shapiro Diskon 20% : W=0.9465, p=0.0000
Shapiro Cashback   : W=0.9351, p=0.0000
Mann-Whitney U : 189263
p-value        : 2.299250e-36
Keputusan      : TOLAK H0
Intercept   : 1582.92
Slope       : 0.00151830
R-squared   : 0.5466
p-value     : 1.440122e-177
Per Rp 1 Jt : 1518 pengunjung
